# 19 — Análise final de dez transcrições

Este notebook seleciona **dez transcrições distintas** do arquivo anonimizado disponível no projeto e chama `analisar_transcricao` do notebook 18 uma vez para cada uma. Ele mostra produto principal e candidatos com fontes, sentimento, risco de churn, oportunidade comercial, termos principais e recomendação de ação. Não é preciso executar manualmente os notebooks anteriores quando os modelos e índices locais já estiverem preparados.

**Como usar:** selecione o kernel `Wedjat (Python 3.12 CUDA)` e clique em **Run All / Executar tudo**. Por padrão, usamos dez registros com IDs de reunião diferentes do arquivo disponível agora. Os índices foram escolhidos entre transcrições de 1.500 a 3.500 caracteres para permitir uma execução local verificável; essa seleção por tamanho não representa uma amostra aleatória. Para usar qualquer outra transcrição ou outro conjunto NDJSON, altere somente a célula de entrada. A análise roda sempre em `full` na GPU, grava o JSON completo em `data/processed/analises_finais_10.json` e apresenta os dados salvos em tabelas pandas.

In [ ]:
# Altere apenas esta célula.
ORIGEM = "arquivo"  # arquivo | texto
TRANSCRICAO_DIRETA = """
"""  # Preencha quando ORIGEM = "texto".
ARQUIVO_ENTRADA = "data/raw/ANON_transcricao.json"  # Arquivo anonimizado disponível no projeto.
INDICES_REGISTROS = [117, 137, 141, 213, 254, 369, 394, 401, 418, 525]  # índices começam em zero.
CAMPO_JSON = None  # Detecta transcricao ou ANON_TRANSCRICAO; informe outro nome se necessário.
ARQUIVO_SAIDA = "data/processed/analises_finais_10.json"  # JSON completo, ignorado pelo Git.

## Fontes e modos

- **Transcrições atuais do projeto:** `ORIGEM="arquivo"`, `ARQUIVO_ENTRADA="data/raw/ANON_transcricao.json"` e os dez índices de `INDICES_REGISTROS`. O arquivo contém 1.174 registros NDJSON; cada índice selecionado tem um ID de reunião diferente. Altere a lista para processar outras reuniões da mesma base.
- **Qualquer outra transcrição:** use `ORIGEM="texto"` e cole o texto entre as aspas triplas de `TRANSCRICAO_DIRETA`; ou informe outro `.txt`, objeto `.json` ou `.jsonl` em `ARQUIVO_ENTRADA`. Para NDJSON, ajuste `INDICES_REGISTROS`. `CAMPO_JSON` informa um campo diferente de `transcricao` ou `ANON_TRANSCRICAO`. Uma entrada de texto, `.txt` ou objeto JSON produz uma análise.
- **Execução:** o modo `full` é fixo neste notebook. Ele exige CUDA, os quatro componentes de modelo e os artefatos locais BERTimbau/E5; não troca silenciosamente para regras. A primeira execução com modelos pode demorar. O JSON é atualizado após cada transcrição e marcado como concluído apenas depois de todas. A última célula relê o arquivo para montar as tabelas pandas.

Cada resultado é um sinal para **revisão humana**. Scores de modelo e similaridade têm interpretações diferentes; nenhum confirma compra ou cancelamento. Confira os produtos candidatos nas fontes. As tabelas omitem o texto integral e o ID; o JSON local contém ambos. Não versione o JSON gerado.

In [ ]:
import json
import time
from pathlib import Path

pasta_atual = Path.cwd().resolve()
candidatos_notebook = [
    pasta_atual / "18_analise_comercial.ipynb",
    pasta_atual / "notebooks" / "18_analise_comercial.ipynb",
]
notebook_pipeline = next((p for p in candidatos_notebook if p.is_file()), None)
if notebook_pipeline is None:
    raise FileNotFoundError("Abra o notebook 19 a partir da raiz do Wedjat ou da pasta notebooks/.")

conteudo_pipeline = json.loads(notebook_pipeline.read_text(encoding="utf-8"))
celulas_definicao = []
for celula in conteudo_pipeline["cells"]:
    if celula["cell_type"] != "code":
        continue
    codigo = "".join(celula["source"])
    if codigo.lstrip().startswith("FONTE_ENTRADA ="):
        break  # Não executa o exemplo nem as saídas do notebook 18.
    celulas_definicao.append(codigo)
if not any(c.lstrip().startswith("def analisar_transcricao(") for c in celulas_definicao):
    raise RuntimeError("A função analisar_transcricao não foi encontrada no notebook 18.")
for codigo in celulas_definicao:
    exec(compile(codigo, str(notebook_pipeline), "exec"), globals())
print("Pipeline da Sprint 4 carregado.")


In [ ]:
def _caminho_entrada(valor):
    caminho = Path(valor).expanduser()
    if caminho.is_absolute():
        return caminho
    opcoes = [pasta_atual / caminho, notebook_pipeline.parent.parent / caminho]
    return next((p for p in opcoes if p.is_file()), opcoes[0])

def _texto_do_objeto(objeto, origem):
    if not isinstance(objeto, dict):
        raise ValueError("O registro deve ser um objeto JSON com uma transcrição.")
    campo = CAMPO_JSON or next(
        (nome for nome in ("transcricao", "ANON_TRANSCRICAO") if nome in objeto), None
    )
    if campo is None or campo not in objeto:
        raise ValueError("Campo da transcrição ausente; configure CAMPO_JSON.")
    return _validar_texto_transcricao(objeto[campo], origem)

def _carregar_arquivo_final(valor):
    caminho = _caminho_entrada(valor)
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo de transcrição não encontrado: {caminho}")
    if caminho.suffix.casefold() == ".txt":
        return [{"indice_registro": None, "id_meeting": None,
                 "transcricao": carregar_transcricao(arquivo=caminho)}]
    if caminho.suffix.casefold() not in {".json", ".jsonl"}:
        raise ValueError("Use .txt, .json ou .jsonl para a transcrição.")
    with caminho.open(encoding="utf-8") as arquivo:
        primeira = arquivo.readline()
        segunda = arquivo.readline()
    ndjson = caminho.suffix.casefold() == ".jsonl" or (
        primeira.rstrip().endswith("}") and segunda.lstrip().startswith("{")
    )
    if ndjson:
        if (not isinstance(INDICES_REGISTROS, (list, tuple))
                or not INDICES_REGISTROS
                or any(type(i) is not int or i < 0 for i in INDICES_REGISTROS)
                or len(set(INDICES_REGISTROS)) != len(INDICES_REGISTROS)):
            raise ValueError("INDICES_REGISTROS deve conter índices inteiros, únicos e não negativos.")
        desejados = set(INDICES_REGISTROS)
        encontrados = {}
        with caminho.open(encoding="utf-8") as arquivo:
            for indice, linha in enumerate(linha for linha in arquivo if linha.strip()):
                if indice not in desejados:
                    continue
                try:
                    objeto = json.loads(linha)
                except json.JSONDecodeError as erro:
                    raise ValueError(f"JSON inválido no registro {indice}.") from erro
                encontrados[indice] = {
                    "indice_registro": indice,
                    "id_meeting": objeto.get("ID_MEETING"),
                    "transcricao": _texto_do_objeto(objeto, f"registro {indice}"),
                }
                if len(encontrados) == len(desejados):
                    break
        ausentes = desejados - encontrados.keys()
        if ausentes:
            raise IndexError(f"Registros ausentes no arquivo: {sorted(ausentes)}")
        registros = [encontrados[i] for i in INDICES_REGISTROS]
        ids = [item["id_meeting"] for item in registros if item["id_meeting"] is not None]
        if len(ids) != len(set(ids)):
            raise ValueError("Os índices selecionados incluem IDs de reunião duplicados.")
        return registros
    try:
        objeto = json.loads(caminho.read_text(encoding="utf-8"))
    except json.JSONDecodeError as erro:
        raise ValueError(f"JSON inválido em {caminho.name}.") from erro
    return [{"indice_registro": None, "id_meeting": objeto.get("ID_MEETING"),
             "transcricao": _texto_do_objeto(objeto, caminho.name)}]

if ORIGEM == "texto":
    registros = [{"indice_registro": None, "id_meeting": None,
                 "transcricao": carregar_transcricao(texto=TRANSCRICAO_DIRETA)}]
    print("Entrada: texto colado.")
elif ORIGEM == "arquivo":
    if ARQUIVO_ENTRADA is None:
        raise ValueError("Informe ARQUIVO_ENTRADA quando ORIGEM for 'arquivo'.")
    registros = _carregar_arquivo_final(ARQUIVO_ENTRADA)
    print(f"Entrada: {Path(ARQUIVO_ENTRADA).name}; {len(registros)} transcrição(ões).")
else:
    raise ValueError("ORIGEM deve ser 'texto' ou 'arquivo'.")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("O notebook 19 exige PyTorch com CUDA e uma GPU disponível.")
if not ARQUIVO_SAIDA:
    raise ValueError("Informe ARQUIVO_SAIDA para gravar o JSON das análises.")
caminho_saida = Path(ARQUIVO_SAIDA).expanduser()
if not caminho_saida.is_absolute():
    caminho_saida = notebook_pipeline.parent.parent / caminho_saida
MODO_ANALISE = "full"
gpu = torch.cuda.get_device_name(0)
torch.cuda.reset_peak_memory_stats(0)
lote = {
    "schema_version": "1.0",
    "source_file": ARQUIVO_ENTRADA if ORIGEM == "arquivo" else "texto",
    "analysis_mode": MODO_ANALISE,
    "gpu": gpu,
    "total_esperado": len(registros),
    "concluido": False,
    "analises": [],
}
print(f"Analisando {len(registros)} transcrição(ões) em modo full com {gpu}...")
inicio_lote = time.perf_counter()
for numero, registro in enumerate(registros, start=1):
    inicio = time.perf_counter()
    resultado = analisar_transcricao(registro["transcricao"], modo=MODO_ANALISE)
    tempo_segundos = round(time.perf_counter() - inicio, 2)
    assert resultado["transcricao_original"] == registro["transcricao"]
    componentes = resultado["analysis_mode"]["components"]
    if set(componentes) != {"products", "sentiment", "churn", "opportunity"} or any(
        mecanismo != "model" for mecanismo in componentes.values()
    ):
        raise RuntimeError("A análise full não usou todos os quatro componentes de modelo.")
    if resultado["recomendacao_acao"]["revisao_humana"] is not True:
        raise RuntimeError("O resultado não exige revisão humana.")
    lote["analises"].append({
        "indice_registro": registro["indice_registro"],
        "id_meeting": registro["id_meeting"],
        "tempo_segundos": tempo_segundos,
        "resultado": resultado,
    })
    salvar_resultado(lote, caminho_saida)  # Preserva as análises concluídas se a próxima falhar.
    print(f"{numero}/{len(registros)}: registro {registro['indice_registro']}, "
          f"{len(registro['transcricao']):,} caracteres, {tempo_segundos} s.")
lote["concluido"] = True
lote["tempo_total_segundos"] = round(time.perf_counter() - inicio_lote, 2)
lote["pico_gpu_mib"] = round(torch.cuda.max_memory_allocated(0) / 1024**2, 1)
salvar_resultado(lote, caminho_saida)
print(f"Lote concluído em {lote['tempo_total_segundos']} s; pico de GPU: "
      f"{lote['pico_gpu_mib']} MiB.")
print("Revisão humana obrigatória antes de qualquer ação comercial.")

In [ ]:
import pandas as pd
from IPython.display import display

json_salvo = json.loads(caminho_saida.read_text(encoding="utf-8"))
if not json_salvo["concluido"] or len(json_salvo["analises"]) != json_salvo["total_esperado"]:
    raise RuntimeError("O JSON salvo contém um lote incompleto.")
print(f"JSON completo salvo em: {caminho_saida}")

linhas_indicadores = []
linhas_produtos = []
for analise in json_salvo["analises"]:
    resultado = analise["resultado"]
    indice = analise["indice_registro"]
    linhas_indicadores.append({
        "Registro": indice,
        "Caracteres": len(resultado["transcricao_original"]),
        "Produto principal": resultado["produto_identificado"],
        "Sentimento": resultado["sentimento"]["label"],
        "Risco de churn": resultado["risco_churn"]["label"],
        "Oportunidade comercial": resultado["oportunidade_comercial"]["label"],
        "Recomendação de ação": resultado["recomendacao_acao"]["label"],
        "Termos principais": ", ".join(resultado["principais_termos"][:10]),
        "Revisão humana": resultado["recomendacao_acao"]["revisao_humana"],
        "Tempo (s)": analise["tempo_segundos"],
    })
    for item in resultado["produtos_candidatos"]:
        linhas_produtos.append({
            "Registro": indice,
            "Produto candidato": item.get("product"),
            "Score": item.get("score"),
            "Tipo de score": item.get("score_type"),
            "Termos": ", ".join(item.get("matched_terms") or []),
            "Fontes": ", ".join(item.get("sources") or []),
        })
tabela_indicadores = pd.DataFrame(linhas_indicadores)
tabela_produtos = pd.DataFrame(linhas_produtos)
print("Uma linha por transcrição analisada:")
display(tabela_indicadores)
print("Produtos candidatos e fontes:")
display(tabela_produtos)

# A tabela vem do JSON relido, mas omite textos e IDs da visualização.
tabela_json = pd.json_normalize(json_salvo["analises"], sep=".")
tabela_json = tabela_json.drop(
    columns=["id_meeting", "resultado.transcricao_original"], errors="ignore"
)
print("JSON salvo apresentado como tabela pandas (sem texto integral ou IDs):")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 120):
    display(tabela_json)
print("Objetos disponíveis: json_salvo, tabela_indicadores, tabela_produtos e tabela_json.")